<a href="https://colab.research.google.com/github/krishnanithesh22-stack/Malay-Indonesian-Words-Originated-From-Sanskrit-Python-Program/blob/main/Malay_Indonesian_Words_Originated_From_Sanskrit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests, time

def get_wiktionary_words(category, lang_code="en", limit=3000):
    words = []
    url = f"https://en.wiktionary.org/w/api.php"
    headers = {
        "User-Agent": "SanskritMalayResearchBot/1.0 (educational NLP project; contact: youremail@example.com)"
    }
    params = {
        "action": "query",
        "list": "categorymembers",
        "cmtitle": f"Category:{category}",
        "cmlimit": "500",
        "format": "json"
    }

    while True:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        if r.status_code != 200:
            print(f"HTTP error {r.status_code} for category '{category}'")
            print(r.text[:300])
            break

        try:
            data = r.json()
        except requests.exceptions.JSONDecodeError:
            print(f"Non JSON response for category '{category}':")
            print(r.text[:300])
            break

        members = data.get("query", {}).get("categorymembers", [])
        words.extend([m["title"] for m in members if ":" not in m["title"]])

        if "continue" in data and len(words) < limit:
            params["cmcontinue"] = data["continue"]["cmcontinue"]
            time.sleep(0.4)
        else:
            break

    return list(set(words))[:limit]

sanskrit_lemmas = get_wiktionary_words("Sanskrit lemmas", limit=3000)
print(f"Fetched {len(sanskrit_lemmas)} Sanskrit reference words.")

Fetched 3000 Sanskrit reference words.


In [2]:
malay_lemmas = get_wiktionary_words("Malay lemmas", limit=3000)
print(f"Fetched {len(malay_lemmas)} Malay reference words.")

indonesian_lemmas = get_wiktionary_words("Indonesian lemmas", limit=3000)
print(f"Fetched {len(indonesian_lemmas)} Indonesian reference words.")

malay_from_sanskrit = get_wiktionary_words("Malay terms derived from Sanskrit", limit=1000)
print(f"Fetched {len(malay_from_sanskrit)} known Malay words of Sanskrit origin.")

indonesian_from_sanskrit = get_wiktionary_words("Indonesian terms derived from Sanskrit", limit=1000)
print(f"Fetched {len(indonesian_from_sanskrit)} known Indonesian words of Sanskrit origin.")

malay_indonesian_lemmas = list(set(malay_lemmas) | set(indonesian_lemmas))
malay_indonesian_from_sanskrit = list(set(malay_from_sanskrit) | set(indonesian_from_sanskrit))

print(f"\nCombined Malay/Indonesian lemmas: {len(malay_indonesian_lemmas)}")
print(f"Combined known Sanskrit-derived words: {len(malay_indonesian_from_sanskrit)}")

Fetched 3000 Malay reference words.
Fetched 3000 Indonesian reference words.
Fetched 534 known Malay words of Sanskrit origin.
Fetched 1000 known Indonesian words of Sanskrit origin.

Combined Malay/Indonesian lemmas: 5654
Combined known Sanskrit-derived words: 1143


In [3]:
import re
import unicodedata

def normalize_latin(word):

    word = str(word).strip().lower()

    # Unicode normalization
    word = unicodedata.normalize("NFKC", word)

    # Remove punctuation
    word = re.sub(r"[^a-zāīūṛṝḷḹṃṁṅñṇṭḍśṣḥ]", "", word)

    return word


# Malay/Indonesian
ml_pairs = [
    (word, normalize_latin(word))
    for word in malay_indonesian_lemmas
]

# Sanskrit
sk_pairs = [
    (word, normalize_latin(word))
    for word in sanskrit_lemmas
]

MIN_WORD_LENGTH = 4

ml_pairs_clean = [
    (original, normalized)
    for original, normalized in ml_pairs
    if len(normalized) >= MIN_WORD_LENGTH
]

sk_pairs_clean = [
    (original, normalized)
    for original, normalized in sk_pairs
    if len(normalized) >= MIN_WORD_LENGTH
]

print("Malay/Indonesian words:", len(ml_pairs_clean))
print("Sanskrit words:", len(sk_pairs_clean))

Malay/Indonesian words: 5380
Sanskrit words: 0


In [4]:
!pip install indic-transliteration -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 4.6 MB/s eta 0:00:00


In [5]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

def sanskrit_to_latin(word):
    try:
        return transliterate(
            word,
            sanscript.DEVANAGARI,
            sanscript.IAST
        ).lower()
    except:
        return str(word).lower()


sk_pairs = [
    (
        original,
        normalize_latin(sanskrit_to_latin(original))
    )
    for original in sanskrit_lemmas
]

sk_pairs_clean = [
    (original, normalized)
    for original, normalized in sk_pairs
    if len(normalized) >= MIN_WORD_LENGTH
]

print("Sanskrit words after transliteration:", len(sk_pairs_clean))

print("\nExamples:")
for original, normalized in sk_pairs_clean[:20]:
    print(original, "→", normalized)

Sanskrit words after transliteration: 2749

Examples:
आचार → ācāra
अभिसमय → abhisamaya
अयोक् → ayok
गुणक → guṇaka
अर्थालंकार → arthālaṃkāra
अकेश → akeśa
कोष्ठ → koṣṭha
जातकर्मन् → jātakarman
काली → kālī
अश्मन् → aśman
अघ → agha
आवश्यक → āvaśyaka
इण्डरी → iṇḍarī
जनन → janana
जिगांसते → jigāṃsate
अग्निहोत्र → agnihotra
उवाप → uvāpa
कौसल्या → kausalyā
अरथी → arathī
उड्डामरिन् → uḍḍāmarin


In [6]:
import pandas as pd

sanskrit_lookup = {}

for original, normalized in sk_pairs_clean:
    if normalized not in sanskrit_lookup:
        sanskrit_lookup[normalized] = []

    sanskrit_lookup[normalized].append(original)


exact_matches = []

for malay_original, malay_normalized in ml_pairs_clean:

    if malay_normalized in sanskrit_lookup:

        for sanskrit_original in sanskrit_lookup[malay_normalized]:

            exact_matches.append({
                "Malay_Indonesian": malay_original,
                "Sanskrit": sanskrit_original,
                "Malay_Normalized": malay_normalized,
                "Sanskrit_Normalized": malay_normalized,
                "Similarity": 1.000,
                "Match_Type": "EXACT 100%"
            })


exact_df = pd.DataFrame(exact_matches)

print("Number of exact matches:", len(exact_df))

exact_df.head(100)

Number of exact matches: 34


,Malay_Indonesian,Sanskrit,Malay_Normalized,Sanskrit_Normalized,Similarity,Match_Type
0,acala,अचल,acala,acala,1.0,EXACT 100%
1,akta,अक्त,akta,akta,1.0,EXACT 100%
2,Arjuna,अर्जुन,arjuna,arjuna,1.0,EXACT 100%
3,ananta,अनन्त,ananta,ananta,1.0,EXACT 100%
4,asta,अस्त,asta,asta,1.0,EXACT 100%
5,goda,गोद,goda,goda,1.0,EXACT 100%
6,arya,अर्य,arya,arya,1.0,EXACT 100%
7,adas,अदस्,adas,adas,1.0,EXACT 100%
8,antar,अन्तर्,antar,antar,1.0,EXACT 100%
9,Arya,अर्य,arya,arya,1.0,EXACT 100%


In [7]:
exact_df = exact_df.drop_duplicates(
    subset=[
        "Malay_Indonesian",
        "Sanskrit"
    ]
).reset_index(drop=True)

print("Unique exact matches:", len(exact_df))

exact_df

Unique exact matches: 34


,Malay_Indonesian,Sanskrit,Malay_Normalized,Sanskrit_Normalized,Similarity,Match_Type
0,acala,अचल,acala,acala,1.0,EXACT 100%
1,akta,अक्त,akta,akta,1.0,EXACT 100%
2,Arjuna,अर्जुन,arjuna,arjuna,1.0,EXACT 100%
3,ananta,अनन्त,ananta,ananta,1.0,EXACT 100%
4,asta,अस्त,asta,asta,1.0,EXACT 100%
5,goda,गोद,goda,goda,1.0,EXACT 100%
6,arya,अर्य,arya,arya,1.0,EXACT 100%
7,adas,अदस्,adas,adas,1.0,EXACT 100%
8,antar,अन्तर्,antar,antar,1.0,EXACT 100%
9,Arya,अर्य,arya,arya,1.0,EXACT 100%


In [8]:
!pip install python-Levenshtein jellyfish -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.7/158.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 42.7 MB/s eta 0:00:00


In [9]:
import jellyfish, Levenshtein

def similarity_score(w1, w2):

  lev_ratio = Levenshtein.ratio(w1, w2)
  jw = jellyfish.jaro_winkler_similarity(w1, w2)
  try:

    soundex_match = int(jellyfish.soundex(w1) == jellyfish.soundex(w2))

  except Exception:

    soundex_match = 0

  combined = (0.4 * lev_ratio)  + (0.5 * jw) + (0.1 * soundex_match)
  return round(lev_ratio, 3), round(jw, 3), soundex_match, round(combined, 3)

In [10]:
from collections import defaultdict

def build_blocks(pairs):

  blocks = defaultdict(list)

  for orig, norm in pairs:

    if norm:

      blocks[norm[0]].append((orig, norm))

  return blocks

sk_blocks = build_blocks(sk_pairs_clean)
print(f"Built {len(sk_blocks)} blocks from Sanskrit words.")



Built 14 blocks from Sanskrit words.


In [13]:
import pandas as pd

def detect_sanskrit_origin_fuzzy(ml_pairs, sk_blocks, threshold = 0.75):

    results = []
    for ml_orig, ml_norm in ml_pairs:

        candidates = sk_blocks.get(ml_norm[0], [])

        if not candidates:
            continue

        best = None

        for sk_orig, sk_norm in candidates:

            lev_ratio, jw, soundex_match, combined = similarity_score(ml_norm, sk_norm)

            if best is None or combined > best[-1]:
                best = (ml_orig, ml_norm, sk_orig, sk_norm, lev_ratio, jw, soundex_match, combined)

        if best and best[-1] >= threshold:
            results.append(best)

    cols = ["Malay_Indonesian", "Malay_Normalized", "Sanskrit",
            "Sanskrit_Normalized", "Levenshtein_Ratio", "Jaro_Winkler",
            "Soundex_Match", "Similarity"]

    df = pd.DataFrame(results, columns=cols)
    if not df.empty:
        df = df.sort_values("Similarity", ascending=False)
    return df

In [15]:
fuzzy_df = detect_sanskrit_origin_fuzzy(ml_pairs_clean, sk_blocks, threshold=0.75)
fuzzy_df["Match_Type"] = "FUZZY"
print(f"Fuzzy matches found: {len(fuzzy_df)}")
fuzzy_df.head(1200)

Fuzzy matches found: 1175


,Malay_Indonesian,Malay_Normalized,Sanskrit,Sanskrit_Normalized,Levenshtein_Ratio,Jaro_Winkler,Soundex_Match,Similarity,Match_Type
1174,anta,anta,अन्त,anta,1.000,1.0,1,1.00,FUZZY
11,acala,acala,अचल,acala,1.000,1.0,1,1.00,FUZZY
1085,ambi-,ambi,अम्बि,ambi,1.000,1.0,1,1.00,FUZZY
425,Anas,anas,अनस्,anas,1.000,1.0,1,1.00,FUZZY
405,ambara,ambara,अम्बर,ambara,1.000,1.0,1,1.00,FUZZY
...,...,...,...,...,...,...,...,...,...
638,akustika,akustika,अक्षिकूट,akṣikūṭa,0.625,0.8,1,0.75,FUZZY
778,abu terbang,abuterbang,अपुत्र,aputra,0.625,0.8,1,0.75,FUZZY
1097,artistik,artistik,अरौत्सीत्,arautsīt,0.625,0.8,1,0.75,FUZZY
410,andurabian,andurabian,ऐन्द्र,aindra,0.625,0.8,1,0.75,FUZZY
